# <font color="#418FDE" size="6.5" uppercase>**Signalmodelle**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Erzeugen Fenster und Statistik- sowie Frequenzmerkmale aus kleinen Signalen. 
- Trainieren scikit-learn-Pipelines für Signal-Klassifikation oder Regression. 
- Bewerten Signalmodelle zeitlich korrekt und vergleichen sie mit Baselines. 


## **1. Signalmerkmale**

### **1.1. Gelabelte Signalfenster**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_01_01.jpg?v=1787651997" width="250">



>* Signale in kurze, lernbare Fenster teilen
>* Labels machen Fenster zu Trainingsbeispielen

>* Fensterlänge bestimmt Kontext und Zustandsklarheit
>* Überlappung hilft, kann Bewertung verfälschen

>* Labels entstehen aus Protokollen, Markierungen oder Ereignissen
>* Übergänge brauchen klare Zuordnungsregeln



In [ ]:
#@title Python-Code - Gelabelte Signalfenster

# Dieses Beispiel erzeugt gelabelte Fenster.
# Es zeigt Mittelwert und Energie.
# Die Grafik markiert Fensterlabels sichtbar.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen ein kleines künstliches Sensorsignal.
sample_rate_hz = 20
seconds = 12
time_s = np.arange(0, seconds, 1 / sample_rate_hz)

# Zwei Aktivitätsphasen erhalten unterschiedliche Schwingungen.
slow_part = np.sin(2 * np.pi * 1.0 * time_s[:120])
fast_part = 1.4 * np.sin(2 * np.pi * 3.0 * time_s[120:])

# Ein fester Zufallsgenerator macht das Beispiel reproduzierbar.
rng = np.random.default_rng(42)
noise = rng.normal(0, 0.15, size=time_s.size)
signal = np.concatenate([slow_part, fast_part]) + noise

# Zu jedem Messpunkt gehört ein bekanntes Zeitlabel.
point_labels = np.where(time_s < 6, "langsam", "schnell")

# Fensterlänge und Schrittweite bestimmen die Beispiele.
window_size = 40
step_size = 20
starts = np.arange(0, len(signal) - window_size + 1, step_size)

# Diese Prüfung schützt vor unpassenden Fensterparametern.
if len(starts) == 0:
    raise ValueError("Die Fensterparameter erzeugen kein einziges Fenster.")

# Jedes Fenster bekommt das dominierende Punktlabel.
rows = []
for start in starts:
    end = start + window_size
    window = signal[start:end]
    labels = point_labels[start:end]
    label = pd.Series(labels).mode().iloc[0]
    rows.append([start, end, label, window.mean(), np.mean(window ** 2)])

# Die Tabelle enthält einfache Merkmale pro Fenster.
columns = ["start", "ende", "label", "mittelwert", "energie"]
features = pd.DataFrame(rows, columns=columns)

# Wir zeigen nur wenige Zeilen für gute Lesbarkeit.
print("Anzahl gelabelter Fenster:", len(features))
print("Fensterlänge in Sekunden:", round(window_size / sample_rate_hz, 1))
print(features.head(5).round(3).to_string(index=False))

# Die Grafik zeigt Signal, Fenstergrenzen und Fensterlabels.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_s, signal, color="black", linewidth=1, label="Signal")

# Farbige Bereiche machen die gelabelten Fenster sichtbar.
colors = {"langsam": "tab:blue", "schnell": "tab:orange"}
for row in features.itertuples(index=False):
    ax.axvspan(row.start / sample_rate_hz, row.ende / sample_rate_hz,
               color=colors[row.label], alpha=0.12)

# Achsenbeschriftungen nennen Zeit und Messwert.
ax.set_title("Gelabelte Signalfenster aus einem Sensorsignal")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Signalwert")
ax.legend(loc="upper right")
plt.show()



### **1.2. Statistische Signalmerkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_01_02.jpg?v=1787651999" width="250">



>* Kennzahlen beschreiben Lage, Streuung und Ausreißer
>* Kompakte Merkmale unterstützen Modelle und Überwachung

>* Wähle einfache, interpretierbare Merkmale passend zur Aufgabe
>* Prüfe komplexe Kennzahlen auf Stabilität und Nutzen

>* Fensterlänge steuert Detail und Stabilität
>* Nur verfügbare Daten für Merkmale nutzen



In [ ]:
#@title Python-Code - Statistische Signalmerkmale

# Dieses Beispiel berechnet statistische Merkmale aus Signalfenstern.
# Kurze Fenster machen lokale Signalunterschiede sichtbar.
# Die Ausgabe vergleicht Ruhe und Bewegung.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen ein kleines synthetisches Beschleunigungssignal.
sample_rate_hz = 20
seconds = 12
sample_count = sample_rate_hz * seconds

# Eine feste Zufallsquelle macht das Beispiel reproduzierbar.
rng = np.random.default_rng(42)
time_s = np.arange(sample_count) / sample_rate_hz
signal = np.empty(sample_count)

# Die erste Hälfte ist ruhig, die zweite stärker schwankend.
quiet_part = rng.normal(loc=0.0, scale=0.05, size=sample_count // 2)
active_wave = 0.8 * np.sin(2 * np.pi * 2.0 * time_s[sample_count // 2:])
active_noise = rng.normal(loc=0.0, scale=0.18, size=sample_count // 2)

# Beide Abschnitte werden zu einem Signal verbunden.
signal[: sample_count // 2] = quiet_part
signal[sample_count // 2:] = active_wave + active_noise

# Wir prüfen die erwartete Signallänge.
if signal.size != sample_count:
    raise ValueError("Das Signal hat nicht die erwartete Länge.")

# Jedes Fenster enthält zwei Sekunden Messwerte.
window_size = 2 * sample_rate_hz
window_count = signal.size // window_size
feature_rows = []

# Für jedes Fenster berechnen wir einfache Statistikmerkmale.
for window_index in range(window_count):
    start = window_index * window_size
    stop = start + window_size
    window = signal[start:stop]
    feature_rows.append(
        {
            "Fenster": window_index + 1,
            "Mittelwert": np.mean(window),
            "Std": np.std(window),
            "Minimum": np.min(window),
            "Maximum": np.max(window),
        }
    )

# Die Merkmale werden als kleine Tabelle lesbar gemacht.
features = pd.DataFrame(feature_rows)
rounded_features = features.round(3)

print("Statistische Merkmale pro 2-Sekunden-Fenster:")
print(rounded_features.to_string(index=False))
print("Hohe Standardabweichung zeigt hier stärkere Bewegung.")

# Die Grafik zeigt Rohsignal und Fenstergrenzen gemeinsam.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_s, signal, label="Beschleunigung", color="tab:blue")

# Vertikale Linien markieren die berechneten Fenster.
for boundary in range(1, window_count):
    ax.axvline(boundary * 2, color="gray", alpha=0.35, linewidth=1)

ax.set_title("Synthetisches Signal mit 2-Sekunden-Fenstern")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Beschleunigung in m/s²")
ax.legend(loc="upper left")
plt.show()



### **1.3. Frequenzen aus Signalen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_01_03.jpg?v=1787651995" width="250">



>* Frequenzen zeigen wiederkehrende Signalanteile.
>* Rhythmen unterscheiden ähnlich wirkende Signale.

>* Frequenzen zeigen langsame und schnelle Signalanteile
>* Fensterlänge beeinflusst robuste Frequenzmerkmale

>* Frequenzmuster zeigen Zustände, Aktivitäten und Störungen
>* Vorbereitung, Kontext und Datenqualität sind entscheidend



In [ ]:
#@title Python-Code - Frequenzen aus Signalen

# Dieses Beispiel zeigt Frequenzen in kleinen Signalfenstern.
# Eine FFT macht Schwingungsanteile als Merkmale sichtbar.
# Die stärksten Frequenzen werden berechnet und geplottet.

import numpy as np
import matplotlib.pyplot as plt

# Wir legen eine bekannte Abtastrate und Fensterlänge fest.
sampling_rate_hz = 100
window_seconds = 2
sample_count = sampling_rate_hz * window_seconds

# Die Zeitachse beschreibt jeden Messpunkt im Fenster.
time_seconds = np.arange(sample_count) / sampling_rate_hz

# Das Signal mischt langsame und schnelle Schwingungen.
slow_part = 1.0 * np.sin(2 * np.pi * 3 * time_seconds)
fast_part = 0.45 * np.sin(2 * np.pi * 12 * time_seconds)
signal = slow_part + fast_part

# Diese Prüfung schützt vor unpassenden Fenstergrößen.
if signal.size != sample_count:
    raise ValueError("Das Signalfenster hat nicht die erwartete Länge.")

# Die FFT zerlegt das Fenster in Frequenzanteile.
frequency_bins = np.fft.rfftfreq(sample_count, d=1 / sampling_rate_hz)
spectrum = np.fft.rfft(signal)
amplitudes = np.abs(spectrum) / sample_count * 2

# Der Gleichanteil wird für die dominante Frequenz ignoriert.
nonzero_amplitudes = amplitudes.copy()
nonzero_amplitudes[0] = 0
peak_index = int(np.argmax(nonzero_amplitudes))

# Wir berechnen einfache Frequenzmerkmale aus dem Spektrum.
dominant_frequency_hz = frequency_bins[peak_index]
low_band_energy = np.sum(amplitudes[(frequency_bins >= 1) & (frequency_bins <= 5)] ** 2)
high_band_energy = np.sum(amplitudes[(frequency_bins > 5) & (frequency_bins <= 20)] ** 2)

print("Frequenzmerkmale aus einem kleinen Signalfenster:")
print(f"Dominante Frequenz: {dominant_frequency_hz:.1f} Hz")
print(f"Energie 1 bis 5 Hz: {low_band_energy:.2f}")
print(f"Energie über 5 bis 20 Hz: {high_band_energy:.2f}")

# Die Grafik zeigt, welche Frequenzen stark vertreten sind.
fig, ax = plt.subplots(figsize=(8, 4))
ax.stem(frequency_bins, amplitudes, basefmt=" ")
ax.set_xlim(0, 20)
ax.set_title("Frequenzspektrum eines synthetischen Signals")

ax.set_xlabel("Frequenz in Hz")
ax.set_ylabel("Amplitude")
ax.axvline(dominant_frequency_hz, color="red", linestyle="--", label="dominant")
ax.legend()

plt.show()



## **2. Zeitlich korrekt trainieren**

### **2.1. Merkmale als Tabelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_02_01.jpg?v=1787651989" width="250">



>* Signale in Beobachtungsfenster zerlegen
>* Merkmale als Tabellenspalten für Modelle nutzen

>* Jedes Fenster braucht einen passenden Zielwert
>* Keine zukünftigen Messwerte in Merkmalen nutzen

>* Merkmalstabellen ermöglichen saubere Pipeline-Schritte
>* Herkunftsdaten verhindern unrealistisch gute Bewertungen



In [ ]:
#@title Python-Code - Merkmale als Tabelle

# Wir erzeugen Merkmale aus kurzen Signalfenstern.
# Eine Pipeline trainiert daraus ein Klassifikationsmodell.
# Die Ausgabe zeigt Tabelle und zeitliche Bewertung.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Ein künstliches Signal wechselt später seine Frequenz.
sample_rate = 50
seconds = 24
time = np.arange(seconds * sample_rate) / sample_rate

# Deterministisches Rauschen macht die Aufgabe realistischer.
rng = np.random.default_rng(42)
noise = rng.normal(0, 0.25, size=time.size)

# Klasse null ist langsam, Klasse eins ist schneller.
labels_per_sample = (time >= 12).astype(int)
frequency = np.where(labels_per_sample == 0, 2.0, 7.0)
signal = np.sin(2 * np.pi * frequency * time) + noise

# Jedes Fenster wird zu einer Tabellenzeile.
window_size = 100
step_size = 50
rows = []

for start in range(0, len(signal) - window_size + 1, step_size):
    end = start + window_size
    window = signal[start:end]
    window_label = int(labels_per_sample[end - 1])

    spectrum = np.abs(np.fft.rfft(window))
    frequencies = np.fft.rfftfreq(window_size, d=1 / sample_rate)
    dominant_frequency = frequencies[1:][np.argmax(spectrum[1:])]

    rows.append(
        {
            "start_s": start / sample_rate,
            "mean": window.mean(),
            "std": window.std(),
            "energy": np.mean(window ** 2),
            "dominant_hz": dominant_frequency,
            "target": window_label,
        }
    )

feature_table = pd.DataFrame(rows)

# Eine einfache Prüfung schützt vor falschen Fenstergrößen.
if feature_table.shape[0] < 6:
    raise ValueError("Zu wenige Fenster für Training und Test.")

# Die zeitliche Aufteilung nutzt frühe Fenster zum Trainieren.
feature_columns = ["mean", "std", "energy", "dominant_hz"]
train_mask = feature_table["start_s"] < 12
X_train = feature_table.loc[train_mask, feature_columns]
y_train = feature_table.loc[train_mask, "target"]

X_test = feature_table.loc[~train_mask, feature_columns]
y_test = feature_table.loc[~train_mask, "target"]

# Die Pipeline skaliert nur mit Trainingsdaten.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=200),
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Fenster insgesamt: {len(feature_table)}")
print(f"Train/Test zeitlich: {len(X_train)}/{len(X_test)}")
print(f"Testgenauigkeit: {accuracy:.2f}")
print("Erste Merkmalstabelle:")
print(feature_table.head(3).round(2).to_string(index=False))

# Die Grafik zeigt, wie Fenster zu Tabellenzeilen werden.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(feature_table["start_s"], feature_table["dominant_hz"], marker="o")
ax.axvline(12, color="black", linestyle="--", label="Train/Test-Grenze")
ax.set_title("Dominante Frequenz pro Signalfenster")
ax.set_xlabel("Fensterstart in Sekunden")
ax.set_ylabel("Dominante Frequenz in Hz")
ax.legend()
plt.show()



### **2.2. Zeitlicher Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_02_02.jpg?v=1787651993" width="250">



>* Zeitlich trennen statt zufällig mischen
>* So vermeidest du unrealistisch gute Testergebnisse

>* Chronologisch trainieren, später validieren oder testen
>* Robustheit gegen Trends und Drift prüfen

>* Überlappende Fenster und Testinformationen vermeiden
>* Validierung und fortschreitende Auswertung nutzen



In [ ]:
#@title Python-Code - Zeitlicher Split

# Dieses Beispiel zeigt einen zeitlichen Split.
# Wir trainieren eine einfache Signal-Pipeline.
# Der Test nutzt spätere, ungesehene Fenster.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Ein künstliches Signal enthält zwei zeitliche Phasen.
rng = np.random.default_rng(42)
sample_count = 240
window_size = 20

# Die Zeitachse bleibt während des gesamten Beispiels sortiert.
time_index = np.arange(sample_count)
base_signal = np.sin(time_index / 6)
noise = rng.normal(0, 0.25, sample_count)

# Spätere Werte haben häufiger die Klasse eins.
signal = base_signal + noise + (time_index >= 120) * 0.8
labels = (signal > 0.45).astype(int)

# Aus jedem Fenster entstehen einfache Statistikmerkmale.
feature_rows = []
window_labels = []
window_times = []

for start in range(sample_count - window_size + 1):
    window = signal[start:start + window_size]
    feature_rows.append([window.mean(), window.std(), window.max()])
    window_labels.append(labels[start + window_size - 1])
    window_times.append(start + window_size - 1)

# Die Merkmalsmatrix passt zu den Fensterlabels.
X = np.array(feature_rows)
y = np.array(window_labels)
window_times = np.array(window_times)

if X.shape[0] != y.shape[0]:
    raise ValueError("Jedes Fenster braucht genau ein Label.")

# Der Split trennt frühere und spätere Fenster.
split_index = int(0.7 * len(X))
X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]
test_times = window_times[split_index:]

# Skalierung und Modell werden nur am Training gelernt.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=200)
)
model.fit(X_train, y_train)

# Die Bewertung nutzt ausschließlich spätere Fenster.
predicted = model.predict(X_test)
accuracy = accuracy_score(y_test, predicted)
majority_class = np.bincount(y_train).argmax()

# Eine einfache Baseline sagt immer die Trainingsmehrheit voraus.
baseline = np.full_like(y_test, majority_class)
baseline_accuracy = accuracy_score(y_test, baseline)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsfenster: {len(X_train)}, Testfenster: {len(X_test)}")
print(f"Zeitlicher Split bei Fensterzeit: {window_times[split_index]}")
print(f"Pipeline-Genauigkeit: {accuracy:.2f}")
print(f"Baseline-Genauigkeit: {baseline_accuracy:.2f}")

# Die Grafik markiert den späteren Testzeitraum.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(window_times, y, label="wahre Klasse", color="black", linewidth=1)
ax.scatter(test_times, predicted, label="Vorhersage im Test", s=18)
ax.axvline(window_times[split_index], color="red", linestyle="--", label="Split")

ax.set_title("Zeitlicher Split für Signal-Fenster")
ax.set_xlabel("Zeitindex des Fensterendes")
ax.set_ylabel("Klasse")
ax.legend()
plt.show()



### **2.3. Pipeline trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_02_03.jpg?v=1787651991" width="250">



>* Pipeline bündelt Skalierung, Auswahl und Modelltraining
>* Gelernte Schritte gelten unverändert für neue Signale

>* Keine Zukunftsdaten ins Training lassen
>* Vorverarbeitung nur am Trainingsanteil anpassen

>* Pipeline-Komponenten gemeinsam auswählen und bewerten
>* Varianten fair zeitlich vergleichen



In [ ]:
#@title Python-Code - Pipeline trainieren

# Wir trainieren eine Pipeline für Signalfenster.
# Skalierung lernt nur aus frühen Trainingsdaten.
# Die spätere Testphase bewertet das Modell fair.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Ein Zufallsgenerator macht das Beispiel reproduzierbar.
rng = np.random.default_rng(42)

# Wir erzeugen kleine Signalfenster mit zwei Zuständen.
n_windows = 240
window_size = 40
sample_index = np.arange(window_size)

# Spätere Fenster haben eine leichte Drift wie reale Sensoren.
time_drift = np.linspace(0.0, 0.8, n_windows)
labels = (np.arange(n_windows) % 2).astype(int)
signals = np.zeros((n_windows, window_size))

# Jedes Fenster bekommt Frequenz, Drift und Rauschen.
for i in range(n_windows):
    frequency = 1.0 + labels[i] * 1.4
    phase = 2.0 * np.pi * frequency * sample_index / window_size
    signals[i] = np.sin(phase) + time_drift[i]
    signals[i] = signals[i] + rng.normal(0.0, 0.35, window_size)

# Aus jedem Fenster entstehen einfache Signalmerkmale.
mean_feature = signals.mean(axis=1)
std_feature = signals.std(axis=1)
energy_feature = np.mean(signals * signals, axis=1)
zero_crossings = np.mean(signals[:, 1:] * signals[:, :-1] < 0, axis=1)

# Die Merkmale bilden die Tabelle für scikit-learn.
features = np.column_stack(
    [mean_feature, std_feature, energy_feature, zero_crossings]
)

# Eine kurze Prüfung schützt vor falschen Formen.
if features.shape != (n_windows, 4):
    raise ValueError("Die Merkmaltabelle hat nicht die erwartete Form.")

# Zeitlich korrekt bedeutet: früh trainieren, später testen.
split_index = 170
X_train = features[:split_index]
y_train = labels[:split_index]
X_test = features[split_index:]
y_test = labels[split_index:]

# Die Pipeline lernt Skalierung und Modell nur am Trainingsteil.
model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=300, random_state=42)),
    ]
)

# Ein einziges fit trainiert alle Pipeline-Schritte gemeinsam.
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# Eine einfache Baseline sagt immer die häufigste Trainingsklasse.
majority_class = int(np.bincount(y_train).argmax())
baseline_predictions = np.full_like(y_test, majority_class)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit der Pipeline: {accuracy:.2f}")
print(f"Testgenauigkeit der Baseline: {baseline_accuracy:.2f}")
print("Skalierung wurde nur mit den frühen Trainingsfenstern gelernt.")

# Die Grafik zeigt Vorhersagen in der späteren Testphase.
fig, ax = plt.subplots(figsize=(8, 4))
test_time = np.arange(split_index, n_windows)
ax.plot(test_time, y_test, label="Wahre Klasse", linewidth=2)
ax.scatter(test_time, predictions, label="Pipeline-Vorhersage", s=24)
ax.set_title("Zeitlich korrekte Pipeline-Bewertung")
ax.set_xlabel("Fensterindex in zeitlicher Reihenfolge")
ax.set_ylabel("Signalzustand")
ax.set_yticks([0, 1])
ax.legend()
plt.show()



## **3. Signalmodelle vergleichen**

### **3.1. Zeitreihen korrekt aufteilen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_03_01.jpg?v=1787651984" width="250">



>* Zeitliche Reihenfolge verhindert Zukunftsinformation im Test
>* Zufällige Splits überschätzen oft die Modellleistung

>* Früher trainieren, später validieren und testen
>* Überlappende Fenster nicht zufällig mischen

>* Rollierende Validierung prüft zeitliche Stabilität
>* Baselines fair unter gleichen Zeitbedingungen vergleichen



In [ ]:
#@title Python-Code - Zeitreihen korrekt aufteilen

# Wir vergleichen zufällige und zeitliche Aufteilungen.
# Überlappende Fenster können Testdaten zu ähnlich machen.
# Die Ausgabe zeigt realistischere zeitliche Bewertung.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

# Ein synthetisches Signal simuliert langsam wechselnde Zustände.
rng = np.random.default_rng(42)
time_steps = np.arange(900)
trend = np.sin(time_steps / 45)
noise = rng.normal(0, 0.35, size=time_steps.size)
signal = trend + noise

# Die spätere Phase ist absichtlich etwas anders verteilt.
signal[600:] = signal[600:] + 0.45
labels = (signal > 0.25).astype(int)
window_size = 30
step_size = 5

# Aus dem Signal entstehen stark überlappende Fenster.
features = []
targets = []
window_ends = []
for start in range(0, len(signal) - window_size + 1, step_size):
    window = signal[start:start + window_size]
    features.append([window.mean(), window.std(), window[-1] - window[0]])
    targets.append(labels[start + window_size - 1])
    window_ends.append(start + window_size - 1)

# Listen werden in NumPy-Arrays umgewandelt.
X = np.array(features)
y = np.array(targets)
window_ends = np.array(window_ends)

# Eine einfache Prüfung schützt vor unerwarteten Formen.
if X.shape[0] != y.shape[0] or X.shape[1] != 3:
    raise ValueError("Die Fenstermerkmale haben eine unerwartete Form.")

# Zufällige Aufteilung mischt nahe Fenster in beide Datenteile.
indices = rng.permutation(len(y))
split_index = int(0.7 * len(y))
random_train = indices[:split_index]
random_test = indices[split_index:]

# Zeitliche Aufteilung trainiert nur auf früheren Fenstern.
time_train = np.where(window_ends < 600)[0]
time_test = np.where(window_ends >= 600)[0]

# Dasselbe Modell wird unter beiden Bedingungen bewertet.
random_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=300)
)
random_model.fit(X[random_train], y[random_train])
random_accuracy = accuracy_score(y[random_test], random_model.predict(X[random_test]))

# Die Baseline nutzt dieselbe zeitliche Testbedingung.
time_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=300)
)
time_model.fit(X[time_train], y[time_train])
time_accuracy = accuracy_score(y[time_test], time_model.predict(X[time_test]))

# Eine faire Baseline wird nur auf Trainingsdaten angepasst.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X[time_train], y[time_train])
baseline_accuracy = accuracy_score(y[time_test], baseline.predict(X[time_test]))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Fenster insgesamt: {len(y)}, Merkmale pro Fenster: {X.shape[1]}")
print(f"Zufällige Aufteilung, Modellgenauigkeit: {random_accuracy:.2f}")
print(f"Zeitliche Aufteilung, Modellgenauigkeit: {time_accuracy:.2f}")
print(f"Zeitliche Aufteilung, Baseline-Genauigkeit: {baseline_accuracy:.2f}")

# Die Grafik markiert Training und zukünftigen Testbereich.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, signal, label="Signal")
ax.axvspan(0, 599, color="tab:blue", alpha=0.12, label="Training")
ax.axvspan(600, 899, color="tab:orange", alpha=0.12, label="Zukünftiger Test")
ax.set_title("Zeitlich korrekte Aufteilung eines Signals")
ax.set_xlabel("Zeitindex")
ax.set_ylabel("Signalwert")
ax.legend()
plt.show()



### **3.2. Vorhersagen sichtbar machen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_03_02.jpg?v=1787651987" width="250">



>* Vorhersagen als Zeitverlauf mit Zielwerten prüfen
>* Nur zeitlich verfügbare Informationen verwenden

>* Zeitliche Vorhersagen mit Ereignissen vergleichen
>* Fehler, Muster und Praxisnutzen erkennen

>* Baselines machen Modellleistung realistisch vergleichbar
>* Visualisierungen zeigen Mehrwert, Fehler und Grenzen



In [ ]:
#@title Python-Code - Vorhersagen sichtbar machen

# Wir machen Signalvorhersagen zeitlich sichtbar.
# Ein Modell wird mit einer Baseline verglichen.
# Die Grafik zeigt Fehler im Verlauf.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen ein kleines künstliches Sensorsignal.
rng = np.random.default_rng(42)
time = np.arange(240)

signal = np.sin(time / 12) + 0.35 * np.sin(time / 4)
noise = rng.normal(0, 0.12, size=time.size)
values = signal + noise

# Das Ziel ist der jeweils nächste Signalwert.
window_size = 8
features = []
targets = []

for start in range(values.size - window_size):
    features.append(values[start:start + window_size])
    targets.append(values[start + window_size])

features = np.array(features)
targets = np.array(targets)

# Wir prüfen, ob Fenster und Ziele zusammenpassen.
if features.shape[0] != targets.shape[0]:
    raise ValueError("Fenster und Zielwerte passen nicht zusammen.")

# Der Testbereich liegt später in der Zeit.
split_index = 170
X_train = features[:split_index]
y_train = targets[:split_index]

X_test = features[split_index:]
y_test = targets[split_index:]
test_time = time[split_index + window_size:]

# Die Baseline sagt den letzten bekannten Wert voraus.
baseline_pred = X_test[:, -1]

# Die Pipeline lernt nur aus der Vergangenheit.
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

# Wir vergleichen Modell und Baseline mit demselben Testzeitraum.
model_mae = mean_absolute_error(y_test, model_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testpunkte im späteren Zeitraum: {len(y_test)}")
print(f"MAE Modell: {model_mae:.3f}")
print(f"MAE Baseline letzter Wert: {baseline_mae:.3f}")

# Die Kurven zeigen, wo Vorhersagen zeitlich gut passen.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_time, y_test, label="Tatsächlicher nächster Wert", linewidth=2)
ax.plot(test_time, model_pred, label="Ridge-Modell", linewidth=2)
ax.plot(test_time, baseline_pred, label="Baseline: letzter Wert", linestyle="--")

ax.set_title("Signalvorhersagen im zeitlich korrekten Testbereich")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Signalwert")
ax.legend()

plt.show()



### **3.3. Signal Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_B/image_03_03.jpg?v=1787651985" width="250">



>* Zeitlich trennen für faire Modellbewertung
>* Echte Muster statt Datenleckage erkennen

>* Baselines setzen faire Mindestanforderungen
>* Modelle zeitlich und fehlerbezogen prüfen

>* Vorhersagen zeitlich visualisieren und vergleichen
>* Nutzen, Fehler und Grenzen begründet bewerten



In [ ]:
#@title Python-Code - Signal Mini Projekt

# Dieses Mini-Projekt bewertet Signale zeitlich korrekt.
# Baselines zeigen eine faire Mindestleistung.
# Die Grafik vergleicht Vorhersagen im Testzeitraum.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen ein kleines synthetisches Verbrauchssignal.
rng = np.random.default_rng(42)
time_index = np.arange(240)
daily_pattern = 2.0 * np.sin(2 * np.pi * time_index / 24)
trend = 0.01 * time_index
noise = rng.normal(0, 0.35, size=time_index.size)
signal = 10 + daily_pattern + trend + noise

# Aus vergangenen Werten entstehen einfache Fenstermerkmale.
window_size = 6
features = []
targets = []
target_times = []

for end_index in range(window_size, len(signal)):
    window = signal[end_index - window_size:end_index]
    features.append([window[-1], window.mean(), window.std()])
    targets.append(signal[end_index])
    target_times.append(end_index)

features = np.array(features)
targets = np.array(targets)
target_times = np.array(target_times)

# Eine kurze Prüfung macht die Datenform nachvollziehbar.
if features.shape[0] != targets.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung bleibt zeitlich geordnet statt zufällig gemischt.
train_end = int(0.7 * len(targets))
X_train = features[:train_end]
y_train = targets[:train_end]
X_test = features[train_end:]
y_test = targets[train_end:]

# Das Modell lernt nur aus der Vergangenheit.
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=42))
model.fit(X_train, y_train)
model_predictions = model.predict(X_test)

# Die Baseline sagt den Trainingsdurchschnitt voraus.
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

# Eine zweite Baseline schreibt den letzten Fensterwert fort.
last_value_predictions = X_test[:, 0]
model_mae = mean_absolute_error(y_test, model_predictions)
mean_mae = mean_absolute_error(y_test, baseline_predictions)
last_mae = mean_absolute_error(y_test, last_value_predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testzeitraum: Zeitpunkte {target_times[train_end]} bis {target_times[-1]}")
print(f"MAE Ridge-Modell: {model_mae:.2f}")
print(f"MAE Durchschnitts-Baseline: {mean_mae:.2f}")
print(f"MAE Letzter-Wert-Baseline: {last_mae:.2f}")

# Die Zeitachse zeigt, ob Fehler systematisch auftreten.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(target_times[train_end:], y_test, label="Echter Wert", linewidth=2)
ax.plot(target_times[train_end:], model_predictions, label="Ridge-Modell")
ax.plot(target_times[train_end:], last_value_predictions, label="Letzter Wert")
ax.set_title("Zeitlich korrekter Test: Modell gegen Baseline")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Signalwert")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Signalmodelle**</font>


In this lecture, you learned to:
- Erzeugen Fenster und Statistik- sowie Frequenzmerkmale aus kleinen Signalen. 
- Trainieren scikit-learn-Pipelines für Signal-Klassifikation oder Regression. 
- Bewerten Signalmodelle zeitlich korrekt und vergleichen sie mit Baselines. 

In the next Module (Module 14), we will go over 'Neuronale Netze'